# Shadow Cell Validation 001 v2

This notebook is a thin launcher for the registered repository runner. It does not implement independent scientific logic. Formal seeds remain `95311`, `95312`, and `95313`; smoke uses `95301`.

In [ ]:
from pathlib import Path
import subprocess, sys, torch
ROOT = Path('/kaggle/working/mini-cells')
BRANCH = 'codex/shadow-cell-validation-001-v2-amendment'
if not (ROOT / '.git').exists():
    subprocess.run(['git', 'clone', '--branch', BRANCH, '--single-branch', 'https://github.com/ArcheLabs/mini-cells.git', str(ROOT)], check=True)
else:
    subprocess.run(['git', 'fetch', 'origin', BRANCH], cwd=ROOT, check=True)
    subprocess.run(['git', 'switch', BRANCH], cwd=ROOT, check=True)
    subprocess.run(['git', 'pull', '--ff-only', 'origin', BRANCH], cwd=ROOT, check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-e', str(ROOT)], check=True)
print('CUDA:', torch.cuda.is_available())
print('protocol:', (ROOT / 'research/validations/shadow-cell-validation-001-v2-developmental-maturation/protocol.json').read_text())

In [ ]:
subprocess.run([sys.executable, str(ROOT / 'scripts/research/run_shadow_cell_validation_001_v2.py'), '--phase', 'smoke', '--seed', '95301', '--device', 'cuda' if torch.cuda.is_available() else 'cpu'], cwd=ROOT, check=True)

## Formal execution

Formal execution is fail-closed. First run the repository lock helper with the exact canonical checkpoint and all three per-seed dataset files, review the resulting protocol/lock diff, and commit that pre-formal lock. Only then invoke the same runner once per registered seed. The runner writes resumable `results/.../seed-<seed>/result.json`, checkpoints, and figures; with `--push-results`, it publishes curated evidence after each completed seed.

In [ ]:
# Formal execution remains blocked until protocol-lock.json is FROZEN.
CHECKPOINT = '/kaggle/input/canonical/checkpoint.pt'
DATASETS = {seed: f'/kaggle/input/shadow-v2/formal-seed-{seed}.json' for seed in (95311, 95312, 95313)}
PUBLISH_RESULTS = True  # Requires the GITHUB_TOKEN Kaggle Secret.
PUBLISH_BRANCH = 'codex/shadow-cell-validation-001-v2-amendment'
if PUBLISH_RESULTS:
    subprocess.run([sys.executable, str(ROOT / 'scripts/research/publish_shadow_cell_validation_001_v2.py'), '--preflight-only', '--branch', PUBLISH_BRANCH], cwd=ROOT, check=True)
for seed in (95311, 95312, 95313):
    command = [sys.executable, str(ROOT / 'scripts/research/run_shadow_cell_validation_001_v2.py'),
               '--phase', 'formal', '--seed', str(seed), '--device', 'cuda',
               '--checkpoint', CHECKPOINT, '--dataset', DATASETS[seed]]
    if PUBLISH_RESULTS:
        command += ['--push-results', '--publish-branch', PUBLISH_BRANCH]
    subprocess.run(command, cwd=ROOT, check=True)

# If PUBLISH_RESULTS is False, aggregate locally after all formal seeds finish:
# subprocess.run([sys.executable, str(ROOT / 'scripts/research/aggregate_shadow_cell_validation_001_v2.py'),
#                 '--results-root', str(ROOT / 'results/shadow-cell-validation-001-v2-developmental-maturation')],
#                cwd=ROOT, check=True)